# **BIBLIOTECAS**

In [ ]:
!pip install pgmpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 27.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 165.3/165.3 kB 6.7 MB/s eta 0:00:00


In [2]:
import numpy as np
from pgmpy.models import DiscreteBayesianNetwork
from pgmpy.factors.discrete import TabularCPD
from pgmpy.inference import VariableElimination
import itertools

# Fila prioritária
import heapq

/home/tarskizzy/Documentos/Projeto Final IA/lib64/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/tarskizzy/Documentos/Projeto Final IA/lib64/python3.14/site-packages/pgmpy/estimators/__init__.py:4: FutureWarning: `pgmpy.estimators.StructureScore` is deprecated and will be removed in v1.3.0. Use `pgmpy.structure_score` instead.
  from .StructureScore import (


# **Estrutura da Rede**

In [4]:
# Conexões de causa e efeito.
# Os sinais vitais apontam para a 'Gravidade'.
modelo = DiscreteBayesianNetwork([
    ('Febre', 'Gravidade'),
    ('SaturacaoO2', 'Gravidade'),
    ('PressaoArterial', 'Gravidade'),
    ('IdadeDoenca', 'Gravidade'),
    ('FreqCardiaca', 'Gravidade'),
    ('NivelDor', 'Gravidade')
])

# **CPTs Independentes (Nós Raiz)**

In [5]:
# probabilidade de um paciente chegar com esse sintoma.
# variable_card indica a quantidade de estados possíveis (ex: 2 = Não/Sim).

cpd_febre = TabularCPD(variable='Febre', variable_card=2, values=[[0.6], [0.4]])
cpd_pressao = TabularCPD(variable='PressaoArterial', variable_card=2, values=[[0.7], [0.3]])
cpd_idade = TabularCPD(variable='IdadeDoenca', variable_card=2, values=[[0.6], [0.4]])
cpd_freq = TabularCPD(variable='FreqCardiaca', variable_card=2, values=[[0.7], [0.3]])
cpd_sat = TabularCPD(variable='SaturacaoO2', variable_card=3, values=[[0.6], [0.3], [0.1]])
cpd_dor = TabularCPD(variable='NivelDor', variable_card=3, values=[[0.5], [0.3], [0.2]])

# **Gerador Contínuo da CPT de Gravidade**

In [6]:
# Pesos heurísticos inspirados na importância clínica dos sinais vitais
# descrita em protocolos como NEWS e Manchester.

pesos = {
    'Febre': [0, 0.5],
    'SaturacaoO2': [0, 2.0, 4.0],
    'PressaoArterial': [0, 2.0],
    'IdadeDoenca': [0, 1.5],
    'FreqCardiaca': [0, 1.0],
    'NivelDor': [0, 1.0, 2.0]
}

combinacoes = list(itertools.product([0,1], [0,1,2], [0,1], [0,1], [0,1], [0,1,2]))
prob_baixa, prob_media, prob_alta = [], [], []

escore_maximo = (
    max(pesos['Febre']) +
    max(pesos['SaturacaoO2']) +
    max(pesos['PressaoArterial']) +
    max(pesos['IdadeDoenca']) +
    max(pesos['FreqCardiaca']) +
    max(pesos['NivelDor'])
)

for combo in combinacoes:
    # Calcula o escore total do paciente
    escore = (pesos['Febre'][combo[0]] + pesos['SaturacaoO2'][combo[1]] +
             pesos['PressaoArterial'][combo[2]] + pesos['IdadeDoenca'][combo[3]] +
             pesos['FreqCardiaca'][combo[4]] + pesos['NivelDor'][combo[5]])

    # Probabilidade Alta usando função Sigmoide (transição suave)
    # Subtraímos 5 para centralizar a curva sigmoide no escore médio
    centro = escore_maximo / 2

    p_a = 1/(1+np.exp(-(escore-centro)))

    # Probabilidade Baixa decai usando uma função sigmoide com sinal inverso
    p_b = 1/(1+np.exp((escore-centro)))

    # Probabilidade média usando uma função gaussiana (em formato de sino)
    sigma = escore_maximo / 4
    p_m = np.exp(-((escore - centro) ** 2) / (2 * sigma ** 2))


    # Normalização final para garantir que a soma da coluna seja exatamente 1.0
    soma = p_a + p_m + p_b
    prob_alta.append(p_a / soma)
    prob_media.append(p_m / soma)
    prob_baixa.append(p_b / soma)

# Montagem da CPT final
cpd_gravidade = TabularCPD(
    variable='Gravidade', variable_card=3,
    evidence=['Febre', 'SaturacaoO2', 'PressaoArterial', 'IdadeDoenca', 'FreqCardiaca', 'NivelDor'],
    evidence_card=[2, 3, 2, 2, 2, 3],
    values=[prob_baixa, prob_media, prob_alta]
)

modelo.add_cpds(cpd_febre, cpd_sat, cpd_pressao, cpd_idade, cpd_freq, cpd_dor, cpd_gravidade)
modelo.check_model()

True

# **Inferência**

In [7]:
inferencia = VariableElimination(modelo)

# Teste para verificar a suavidade da curva:
res_leve = inferencia.query(variables=['Gravidade'], evidence={'Febre':0, 'SaturacaoO2':0, 'PressaoArterial':0, 'IdadeDoenca':0, 'FreqCardiaca':0, 'NivelDor':0})
res_med = inferencia.query(variables=['Gravidade'], evidence={'Febre':0, 'SaturacaoO2':1, 'PressaoArterial':0, 'IdadeDoenca':1, 'FreqCardiaca':0, 'NivelDor':0})
res_grave = inferencia.query(variables=['Gravidade'], evidence={'Febre':0, 'SaturacaoO2':2, 'PressaoArterial':1, 'IdadeDoenca':1, 'FreqCardiaca':1, 'NivelDor':1})

print("Paciente Saudável (Escore 0):\n", res_leve)
print("\nPaciente Intermediário (Escore 3.5):\n", res_med)
print("\nPaciente Grave (Escore 9.5):\n", res_grave)

Paciente Saudável (Escore 0):
 +--------------+------------------+
| Gravidade    |   phi(Gravidade) |
+==============+==================+
| Gravidade(0) |           0.8772 |
+--------------+------------------+
| Gravidade(1) |           0.1192 |
+--------------+------------------+
| Gravidade(2) |           0.0036 |
+--------------+------------------+

Paciente Intermediário (Escore 3.5):
 +--------------+------------------+
| Gravidade    |   phi(Gravidade) |
+==============+==================+
| Gravidade(0) |           0.4983 |
+--------------+------------------+
| Gravidade(1) |           0.4343 |
+--------------+------------------+
| Gravidade(2) |           0.0674 |
+--------------+------------------+

Paciente Grave (Escore 9.5):
 +--------------+------------------+
| Gravidade    |   phi(Gravidade) |
+==============+==================+
| Gravidade(0) |           0.0134 |
+--------------+------------------+
| Gravidade(1) |           0.2577 |
+--------------+------------------+

# **Gerador de Pacientes**

In [8]:
# Para que os testes sejam sempre iguais
np.random.seed(42)

# Classe objeto para os pacientes
class Paciente:
    def __init__(self, id, febre, saturacao, pressao,
                 idade_doenca, freq_cardiaca, nivel_dor,
                 tempo_espera, risco):

        self.id = id

        # Sintomas
        self.febre = febre
        self.saturacao = saturacao
        self.pressao = pressao
        self.idade_doenca = idade_doenca
        self.freq_cardiaca = freq_cardiaca
        self.nivel_dor = nivel_dor
        self.risco = 0

        # Tempo de espera (minutos)
        self.tempo_espera = tempo_espera

        # Resultados da Rede Bayesiana
        self.prob_baixa = None
        self.prob_media = None
        self.prob_alta = None

    def mostrar_probabilidades(self):
        return (
            f"Paciente {self.id} | "
            f"Baixa={self.prob_baixa:.2f}, "
            f"Média={self.prob_media:.2f}, "
            f"Alta={self.prob_alta:.2f}"
        )

    def __repr__(self):
        return (
            f"Paciente {self.id} | "
            f"Febre={self.febre}, "
            f"Sat={self.saturacao}, "
            f"Pres. Arterial={self.pressao}, "
            f"Idade={self.idade_doenca}, "
            f"Freq. Cardíaca={self.freq_cardiaca}, "
            f"Dor={self.nivel_dor}, "
            f"Espera={self.tempo_espera} min, "
            f"Risco={self.risco:.2f} "
        )

# Função para gerar um paciente (probabilidades iguais aos dos CPDs)
def gerar_paciente(id):

    febre = np.random.choice(
        [0, 1],
        p=[0.6, 0.4]
    )
    saturacao = np.random.choice(
        [0, 1, 2],
        p=[0.6, 0.3, 0.1]
    )

    pressao = np.random.choice(
        [0, 1],
        p=[0.7, 0.3]
    )

    idade_doenca = np.random.choice(
        [0, 1],
        p=[0.6, 0.4]
    )

    freq_cardiaca = np.random.choice(
        [0, 1],
        p=[0.7, 0.3]
    )

    nivel_dor = np.random.choice(
        [0, 1, 2],
        p=[0.5, 0.3, 0.2]
    )

    # Tempo de espera entre 5 e 60 minutos
    tempo_espera = np.random.randint(5, 61)

    risco = None

    return Paciente(
        id,
        febre,
        saturacao,
        pressao,
        idade_doenca,
        freq_cardiaca,
        nivel_dor,
        tempo_espera,
        risco
    )

# Função para gerar um número n de pacientes
def gerar_lista_pacientes(n):

    pacientes = []

    for i in range(1, n + 1):
        pacientes.append(gerar_paciente(i))

    return pacientes

# Função para calcular as probabilidades de gravidade de um paciente
def avaliar_paciente(paciente):

    resultado = inferencia.query(
        variables=["Gravidade"],
        evidence={
            "Febre": paciente.febre,
            "SaturacaoO2": paciente.saturacao,
            "PressaoArterial": paciente.pressao,
            "IdadeDoenca": paciente.idade_doenca,
            "FreqCardiaca": paciente.freq_cardiaca,
            "NivelDor": paciente.nivel_dor
        }
    )

    paciente.prob_baixa = resultado.values[0]
    paciente.prob_media = resultado.values[1]
    paciente.prob_alta = resultado.values[2]
    paciente.risco = paciente.prob_alta * paciente.tempo_espera
    paciente.risco = paciente.risco

    return paciente

# Função para avaliar as probabilidades de gravidades de todos os pacientes
def avaliar_lista_pacientes(lista):

    for paciente in lista:
        avaliar_paciente(paciente)

    return lista


pacientes = gerar_lista_pacientes(30)
avaliar_lista_pacientes(pacientes)

for p in pacientes:
    print(p)
    print(p.mostrar_probabilidades())
    print("-----------------")

Paciente 1 | Febre=0, Sat=2, Pres. Arterial=1, Idade=0, Freq. Cardíaca=0, Dor=0, Espera=15 min, Risco=4.71 
Paciente 1 | Baixa=0.19, Média=0.50, Alta=0.31
-----------------
Paciente 2 | Febre=0, Sat=0, Pres. Arterial=0, Idade=1, Freq. Cardíaca=0, Dor=1, Espera=34 min, Risco=1.04 
Paciente 2 | Baixa=0.61, Média=0.36, Alta=0.03
-----------------
Paciente 3 | Febre=0, Sat=0, Pres. Arterial=0, Idade=0, Freq. Cardíaca=0, Dor=0, Espera=53 min, Risco=0.19 
Paciente 3 | Baixa=0.88, Média=0.12, Alta=0.00
-----------------
Paciente 4 | Febre=0, Sat=0, Pres. Arterial=0, Idade=1, Freq. Cardíaca=0, Dor=0, Espera=55 min, Risco=0.73 
Paciente 4 | Baixa=0.73, Média=0.26, Alta=0.01
-----------------
Paciente 5 | Febre=0, Sat=2, Pres. Arterial=0, Idade=1, Freq. Cardíaca=0, Dor=0, Espera=22 min, Risco=5.50 
Paciente 5 | Baixa=0.25, Média=0.50, Alta=0.25
-----------------
Paciente 6 | Febre=1, Sat=2, Pres. Arterial=1, Idade=0, Freq. Cardíaca=0, Dor=1, Espera=51 min, Risco=25.41 
Paciente 6 | Baixa=0.07, M

# **Implementação do A**

## Construção do Nó em que o A* irá percorrer

In [12]:

# Estrutura interna para os nós do A*
class NoAStar:
    def __init__(self, pacientes_restantes, caminho, g, h):
        self.pacientes_restantes = pacientes_restantes  # frozenset de índices (IDs)
        self.caminho = caminho                          # Lista com a ordem dos IDs atendidos
        self.g = g                                      # Custo real acumulado (risco)
        self.h = h                                      # Estimativa heurística restante
        self.f = g + h                                  # Custo total estimado

    def __lt__(self, outro):
        return self.f < outro.f


## Construindo a função heurística para estimativa de custo

In [13]:
# Cálculo da Heurística
def calcular_heuristica(ids_restantes, passos_atuais,pacientes_dict):
    """
    Heurística Admissível: Ordena os pacientes restantes pelo maior risco.
    Simula o cenário ideal para estimar o custo mínimo restante de forma otimista.
    """
    if not ids_restantes:
        return 0
    
    # Ordena os restantes pelo maior risco (estratégia gulosa ideal)
    ordenados = sorted(
        [pacientes_dict[pid] for pid in ids_restantes],
        key=lambda x: x["risco"],
        reverse=True
    )
    
    custo_h = 0
    passos_simulados = passos_atuais
    restantes_simulados = list(ordenados)
    
    # Simula o esvaziamento da fila no melhor cenário possível
    while len(restantes_simulados) > 1:
        passos_simulados += 1
        restantes_simulados.pop(0) # Atende o de maior risco imediatamente
        
        # Os que continuam esperando acumulam risco baseado no tempo atualizado
        for p in restantes_simulados:
            tempo_atualizado = p["tempo_inicial"] + passos_simulados
            custo_h += p["risco"] * tempo_atualizado
            
    return custo_h



## Implementação do A*

In [14]:
# Implementação do Algoritmo
def a_star_triagem(lista_pacientes):
    """
    Executa o algoritmo A* para encontrar a ordem ótima de atendimento.
    lista_pacientes: Lista de vetores/tuplas no formato (nivel_de_risco, tempo_de_espera)
    """
    # Mapeia a lista para um dicionário indexado para identificar cada paciente unicamente
    pacientes_dict = {
        i: {"risco": vetor[0], "tempo_inicial": vetor[1], "vetor_original": vetor}
        for i, vetor in enumerate(lista_pacientes)
    }
    
    total_pacientes = len(lista_pacientes)
    

    # ==========================================
    # Inicialização do A*
    # ==========================================
    ids_iniciais = frozenset(pacientes_dict.keys())
    h_inicial = calcular_heuristica(ids_iniciais, passos_atuais=0,pacientes_dict=pacientes_dict)
    
    no_inicial = NoAStar(pacientes_restantes=ids_iniciais, caminho=[], g=0, h=h_inicial)
    
    open_list = []
    heapq.heappush(open_list, no_inicial)
    closed_set = set()
    
    # Loop de busca do A*
    while open_list:
        no_atual = heapq.heappop(open_list)
        
        # OBJETIVO: Fila vazia (todos atendidos)
        if not no_atual.pacientes_restantes:
            # Retorna a sequência de vetores original ordenada e o custo final
            ordem_final = [pacientes_dict[pid]["vetor_original"] for pid in no_atual.caminho]
            return ordem_final, no_atual.g
            
        if no_atual.pacientes_restantes in closed_set:
            continue
            
        closed_set.add(no_atual.pacientes_restantes)
        
        # Mede quantos pacientes já foram atendidos até este nó
        passos_atuais = total_pacientes - len(no_atual.pacientes_restantes)
        
        # AÇÃO: Escolher o próximo paciente a ser retirado da fila
        for id_escolhido in no_atual.pacientes_restantes:
            proximos_restantes = no_atual.pacientes_restantes - frozenset([id_escolhido])
            
            # Avança 1 unidade de tempo para o atendimento atual
            proximo_passo = passos_atuais + 1
            
            # CUSTO DA AÇÃO: Risco acumulado de quem CONTINUA esperando
            # Risco do passo = Soma de (Risco * Tempo Atualizado de Espera) para os restantes
            custo_passo = sum(
                pacientes_dict[pid]["risco"] * (pacientes_dict[pid]["tempo_inicial"] + proximo_passo)
                for pid in proximos_restantes
            )
            
            proximo_g = no_atual.g + custo_passo
            proximo_h = calcular_heuristica(proximos_restantes, passos_atuais=proximo_passo,pacientes_dict=pacientes_dict)
            proximo_caminho = no_atual.caminho + [id_escolhido]
            
            novo_no = NoAStar(proximos_restantes, proximo_caminho, proximo_g, proximo_h)
            heapq.heappush(open_list, novo_no)
            
    return None, 0

## Construindo o vetor de pacientes a ser trabalhado e executando o algoritmo

In [ ]:
vetor_pacientes = []
for p in pacientes:
    paciente = (p.risco, p.tempo_espera, p.id) # Vetor paciente, com seu id, para melhor identificação
    vetor_pacientes.append(paciente)


fila_otimizada, custo = a_star_triagem(vetor_pacientes)

print("A Melhor configuração de fila que se pode ter é:")
for p in fila_otimizada:
    print(f"Paciente {p[2]}, risco: {p[0]}, tempo: {p[1]} min")